# 752个未构建案例的本机审计
以下单元读取本轮实际产物，不覆盖原数据。完整逐案例逻辑在 `../scripts/audit_unbuilt_local.py`；隔离重建在 `../scripts/rebuild_audited_cases.py`。

In [1]:
from pathlib import Path
import json
from collections import Counter
ROOT = Path('/root/sitian')
AUDIT = ROOT / 'data/local_audit_20260911'
R = AUDIT / 'results'
rows = json.loads((R/'case_findings.json').read_text())
print('Cases:', len(rows))
print('By month:', dict(Counter(r['issue_date'][:7] for r in rows)))

Cases: 752
By month: {'2025-04': 147, '2025-05': 578, '2026-02': 27}


In [2]:
print('Original builder against present files/old DB:', dict(Counter(r['original_builder_result_now'] for r in rows)))
print('Valid rebuilt base cases:', sum(r['can_rebuild_with_raw_valid_truth'] for r in rows))
print('Remaining blockers:', dict(Counter(b for r in rows for b in r['current_blockers'])))
print('Distinct invalid truth city-days:', len({(r['city'], f['date']) for r in rows for f in r['raw_truth_failures']}))

Original builder against present files/old DB: {'ok': 725, 'obs_insufficient': 27}
Valid rebuilt base cases: 649
Remaining blockers: {'raw_truth_invalid': 76, 'input_observations_below_48_valid_hours': 27}
Distinct invalid truth city-days: 47


In [3]:
files = json.loads((R/'observation_file_inventory.json').read_text())
for r in files:
    if '2026-02-02' <= r['date'] <= '2026-02-06':
        print(r['date'], r['row_counts'], r['missing_city_columns'])

2026-02-02 {'PM2.5': 7, 'PM10': 7, 'SO2': 7, 'NO2': 7, 'O3': 7, 'O3_8h': 7, 'CO': 7} []
2026-02-03 {'PM2.5': 1, 'PM10': 1, 'SO2': 1, 'NO2': 1, 'O3': 1, 'O3_8h': 1, 'CO': 1} []
2026-02-04 {'PM2.5': 2, 'PM10': 2, 'SO2': 2, 'NO2': 2, 'O3': 2, 'O3_8h': 2, 'CO': 2} []
2026-02-05 {'PM2.5': 12, 'PM10': 12, 'SO2': 12, 'NO2': 12, 'O3': 12, 'O3_8h': 12, 'CO': 12} []
2026-02-06 {'PM2.5': 24, 'PM10': 24, 'SO2': 24, 'NO2': 24, 'O3': 24, 'O3_8h': 24, 'CO': 24} []


In [4]:
rebuilt = json.loads((AUDIT/'rebuild/summary.json').read_text())
print(json.dumps(rebuilt, ensure_ascii=False, indent=2))
old = json.loads((R/'previously_built_raw_reaudit.json').read_text())
print('Previously built still invalid:', old['affected_cases'], old['affected_cases_by_split'])

{
  "counts": {
    "ok": 649,
    "truth_incomplete": 76,
    "obs_insufficient": 27
  },
  "samples": 5,
  "corrected_code_tests_passed": true,
  "successful_case_time_gate_failures": 0,
  "all_generated_files_hashed": true,
  "new_daily_db": {
    "path": "/root/sitian/data/local_audit_20260911/rebuild/workspace/data/aq_daily.sqlite",
    "bytes": 1695744,
    "sha256": "1ba9215f6ccd6da25b5147f69865f94673d31b7878ae585795f7ad856de9fc0f",
    "mtime_utc": "2026-09-11T15:26:39.599193+00:00"
  },
  "training_ready": false,
  "training_ready_reason": "None of the 33 issue dates was included in either existing open-evidence manifest; 132 expected derived files absent in each evidence root. Base cases must not be used as full-contract RL data.",
  "original_dataset_changed": false,
  "active_training_changed": false
}
Previously built still invalid: 491 {'train': 444, 'test': 28, 'val': 19}


In [5]:
print(json.dumps(json.loads((R/'nwp_local_cache_summary.json').read_text()), indent=2))
print('Positive path control:', json.loads((R/'nwp_path_positive_control.json').read_text()))
print('Preservation receipt:', json.loads((R/'preservation_verification.json').read_text()))

{
  "synoptic_gfs": {
    "unique_requested_files": 660,
    "present_in_any_checked_root": 0,
    "missing_in_all_checked_roots": 660
  },
  "synoptic_ifs": {
    "unique_requested_files": 660,
    "present_in_any_checked_root": 0,
    "missing_in_all_checked_roots": 660
  },
  "radiation_gfs": {
    "unique_requested_files": 759,
    "present_in_any_checked_root": 0,
    "missing_in_all_checked_roots": 759
  },
  "radiation_ifs": {
    "unique_requested_files": 792,
    "present_in_any_checked_root": 0,
    "missing_in_all_checked_roots": 792
  }
}
Positive path control: [{'product': 'synoptic', 'source': 'gfs', 'issue_date': '2025-04-01', 'requested': 20, 'found': 20}, {'product': 'synoptic', 'source': 'ifs', 'issue_date': '2025-04-01', 'requested': 20, 'found': 20}, {'product': 'radiation', 'source': 'gfs', 'issue_date': '2025-04-01', 'requested': 23, 'found': 23}, {'product': 'radiation', 'source': 'ifs', 'issue_date': '2025-04-01', 'requested': 24, 'found': 24}]
Preservation rece

## 新一轮的保护清单与原池复核
以下可选单元默认不执行写入。应在新重建之前运行，只写新审计目录；不调用原审计CLI。

In [6]:
# Set RUN_NEW to a new audit directory only when making a new protected run.
RUN_NEW = None
if RUN_NEW is not None:
    import sys
    sys.path.insert(0, str(ROOT/'scripts'))
    from audit_unbuilt_local import sha, dump, load_module
    result = Path(RUN_NEW)/'results'
    guard=[]
    for directory in ('cases','configs','data/verl','references'):
        for p in sorted((ROOT/directory).rglob('*')):
            if p.is_file() and '__pycache__' not in p.parts:
                guard.append({'path':str(p),'sha256':sha(p),'bytes':p.stat().st_size})
    for p in (ROOT/'data/interim').glob('*.json'):
        guard.append({'path':str(p),'sha256':sha(p),'bytes':p.stat().st_size})
    dump(result/'frozen_original_guard.json',guard)
    a=load_module(ROOT/'scripts/audit_national_data.py','existing_audit')
    rows_old,days=a.load_cases()
    check=a.audit_raw(days,{x['case_id']:{'split':x['split'],'stratum':x['stratum']} for x in rows_old})
    dump(result/'previously_built_raw_reaudit.json',check)
